# 2일차 1~3·5교시 · Colab에서 테스트 실행하기

fork한 실습 저장소를 Colab으로 내려받아 **unittest, PyTest, 커버리지, 성능 리포트**를 실행합니다.

- 코드 수정과 커밋은 **GitHub 웹 편집기**에서 합니다(저장소 화면에서 `.` 키 → github.dev). 커밋하면 GitHub Actions가 같은 테스트를 자동으로 실행합니다.
- 이 노트북은 수정한 코드가 통과하는지 **커밋 전에 미리 확인**하는 용도입니다.

In [ ]:
# ① 본인 GitHub 아이디로 바꾸세요 (fork한 저장소를 내려받습니다)
GITHUB_USER = "leejaehee1"

import os
REPO = f"https://github.com/{GITHUB_USER}/ai-quality-lab.git"
if not os.path.exists("/content/ai-quality-lab"):
    !git clone -q {REPO} /content/ai-quality-lab
%cd /content/ai-quality-lab/day2
!pip install -q pytest pytest-cov

## 1. unittest로 실행하기 (1교시)

In [ ]:
!PYTHONPATH=src python -m unittest discover -s tests -p "test_*unittest.py" -v

## 2. PyTest로 전체 테스트 실행하기 (2교시)

`-q`는 간단히, `-v`는 테스트 이름까지 자세히 보여 줍니다.

In [ ]:
!python -m pytest -q

## 3. 커버리지 측정하기 (2교시)

`Missing` 열은 한 번도 실행되지 않은 줄 번호입니다.

In [ ]:
!python -m pytest -q --cov=mlqa --cov-report=term-missing

**생각해 보기**: 테스트가 모두 통과하고 커버리지가 98%입니다. 그러면 이 코드에는 결함이 없을까요? (1일차 원칙 1)

## 4. 테스트 추가하기 (2·3교시)

아래 셀은 새 테스트 파일을 만듭니다. `# TODO` 부분을 채우고 실행한 뒤, 다음 셀로 테스트를 돌려 보세요.

- 힌트 1: 1일차 2교시의 **경계값**(2값·3값)을 `parametrize`로
- 힌트 2: 1일차 5교시의 **데이터 누수**: 결측 대체값은 어느 데이터로 계산해야 하나?
- 힌트 3: 1일차 3교시의 **양성 클래스(pos_label)**: 정밀도를 손으로 계산한 값과 비교

In [ ]:
%%writefile tests/test_my_checks.py
import numpy as np
import pandas as pd
import pytest

from mlqa.preprocess import is_eligible, impute_median
from mlqa.evaluate import compute_metrics


@pytest.mark.parametrize("age, expected", [
    (25, True),     # TODO: 경계값을 추가하세요
])
def test_age_boundaries(age, expected):
    assert is_eligible(age, 12, False) == expected


def test_impute_median_uses_train_only():
    train = pd.DataFrame({"x": [1.0, 2.0, np.nan, 4.0]})
    test = pd.DataFrame({"x": [np.nan, 100.0]})
    _, test_filled, medians = impute_median(train, test, ["x"])
    # TODO: 학습 데이터의 중앙값은 얼마여야 하나요? 기대값을 적어 확인하세요
    assert medians["x"] == medians["x"]

In [ ]:
!python -m pytest -q tests/test_my_checks.py

테스트가 **실패**하면 결함을 찾은 것입니다. 실패 메시지를 복사해 두세요. 7교시에 이슈로 보고합니다.

> 코드를 고치기 전에, 먼저 **테스트가 실패하는 것을 확인**하는 것이 좋습니다. 고친 뒤 같은 테스트가 통과하면 결함이 정말 고쳐졌다는 근거가 됩니다(확인 테스트).

## 5. 성능 리포트 생성하기 (5교시)

In [ ]:
!PYTHONPATH=src python -m mlqa.report

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("reports/model_report.md", encoding="utf-8").read().split("![confusion")[0]))
display(Image("reports/confusion_matrix.png"))

**리포트 검토하기**: 표 2의 `precision`과 아래 classification_report의 `malignant` 행 precision이 같은가요? 다르다면 왜일까요?